# 🌍 LST数据提取工具 - 5分钟快速教程

欢迎！这个教程将让你在5分钟内学会使用LST数据提取工具。

## 📚 教程目录

1. **教程1**: 提取北京地表温度（2分钟）
2. **教程2**: 批量处理多个城市（2分钟）
3. **教程3**: 数据可视化入门（1分钟）

---

## 🚀 开始前的准备

### 第一步：导入库

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

import ee
ee.Initialize()

print("✅ 库导入成功，GEE已连接！")

---

## 📖 教程1: 提取北京地表温度

**目标**: 提取北京市2023年的平均地表温度

**时间**: 2分钟

### 步骤1: 定义研究区域

In [ ]:
# 北京的边界框
beijing_bbox = {
    'min_lon': 116.0,
    'min_lat': 39.7,
    'max_lon': 116.8,
    'max_lat': 40.2
}

print("📍 北京边界:")
print(f"   经度: {beijing_bbox['min_lon']}° ~ {beijing_bbox['max_lon']}°")
print(f"   纬度: {beijing_bbox['min_lat']}° ~ {beijing_bbox['max_lat']}°")

### 步骤2: 创建采样点

In [ ]:
# 创建采样点函数
def create_sampling_points(bbox, spacing=0.1):
    """创建规则采样点"""
    lons = np.arange(bbox['min_lon'], bbox['max_lon'], spacing)
    lats = np.arange(bbox['min_lat'], bbox['max_lat'], spacing)
    
    points = []
    for lon in lons:
        for lat in lats:
            points.append({
                'longitude': lon,
                'latitude': lat,
                'geometry': Point(lon, lat)
            })
    
    return gpd.GeoDataFrame(points, crs='EPSG:4326')

# 创建采样点
beijing_points = create_sampling_points(beijing_bbox, spacing=0.1)
print(f"✅ 创建了 {len(beijing_points)} 个采样点")

### 步骤3: 提取LST数据

In [ ]:
# 定义提取函数
def extract_lst_data(points_df, start_date, end_date):
    """提取地表温度数据"""
    
    # 创建FeatureCollection
    features = []
    for idx, row in points_df.iterrows():
        geom = row.geometry
        point = ee.Geometry.Point([geom.x, geom.y])
        feature = ee.Feature(point, {'id': idx})
        features.append(feature)
    
    fc = ee.FeatureCollection(features)
    
    # 获取MODIS LST数据
    collection = ee.ImageCollection('MODIS/061/MOD11A2')\
        .filterDate(start_date, end_date)\
        .select('LST_Day_1km')
    
    # 计算年平均
    lst_mean = collection.mean()
    
    # 提取数据
    results = lst_mean.sampleRegions(
        collection=fc,
        scale=1000
    )
    
    # 转换为DataFrame
    data = results.getInfo()['features']
    
    extracted_data = []
    for item in data:
        props = item['properties']
        extracted_data.append({
            'id': props.get('id'),
            'LST_raw': props.get('LST_Day_1km')
        })
    
    return pd.DataFrame(extracted_data)

# 提取数据
lst_data = extract_lst_data(beijing_points, '2023-01-01', '2023-12-31')
print(f"✅ 成功提取 {len(lst_data)} 个数据点")

### 步骤4: 数据处理和分析

In [ ]:
# 转换为摄氏度
lst_data['LST_Celsius'] = lst_data['LST_raw'] * 0.02 - 273.15

# 添加坐标信息
lst_data['longitude'] = beijing_points.geometry.x.values
lst_data['latitude'] = beijing_points.geometry.y.values

# 显示统计信息
print("📊 北京2023年平均地表温度:")
print(f"   平均温度: {lst_data['LST_Celsius'].mean():.2f}°C")
print(f"   最高温度: {lst_data['LST_Celsius'].max():.2f}°C")
print(f"   最低温度: {lst_data['LST_Celsius'].min():.2f}°C")
print(f"   温度范围: {lst_data['LST_Celsius'].max() - lst_data['LST_Celsius'].min():.2f}°C")

# 显示部分数据
print("\n📋 数据预览:")
print(lst_data[['longitude', 'latitude', 'LST_Celsius']].head())

### 步骤5: 保存数据

In [ ]:
# 保存为CSV
lst_data.to_csv('beijing_lst_2023.csv', index=False)
print("💾 数据已保存到: beijing_lst_2023.csv")

# 保存为GeoJSON
gdf = gpd.GeoDataFrame(
    lst_data,
    geometry=beijing_points.geometry,
    crs='EPSG:4326'
)
gdf.to_file('beijing_lst_2023.geojson', driver='GeoJSON')
print("💾 数据已保存到: beijing_lst_2023.geojson")

---

## 📖 教程2: 批量处理多个城市

**目标**: 提取北京、上海、广州三个城市的LST数据

**时间**: 2分钟

### 步骤1: 定义多个城市

In [ ]:
# 定义多个城市
cities = {
    '北京': {'min_lon': 116.0, 'min_lat': 39.7, 'max_lon': 116.8, 'max_lat': 40.2},
    '上海': {'min_lon': 121.0, 'min_lat': 31.0, 'max_lon': 121.8, 'max_lat': 31.5},
    '广州': {'min_lon': 113.0, 'min_lat': 23.0, 'max_lon': 113.5, 'max_lat': 23.5}
}

print(f"🏙️ 准备处理 {len(cities)} 个城市")

### 步骤2: 批量提取数据

In [ ]:
# 批量提取函数
def batch_extract_cities(cities_dict, start_date, end_date):
    """批量提取多个城市的数据"""
    
    results = {}
    
    for city_name, bbox in cities_dict.items():
        print(f"\n📍 处理 {city_name}...")
        
        # 创建采样点
        points = create_sampling_points(bbox, spacing=0.1)
        
        # 提取数据
        try:
            lst_data = extract_lst_data(points, start_date, end_date)
            
            # 处理数据
            if lst_data is not None and len(lst_data) > 0:
                lst_data['LST_Celsius'] = lst_data['LST_raw'] * 0.02 - 273.15
                lst_data['city'] = city_name
                
                results[city_name] = lst_data
                print(f"   ✅ {city_name}: 平均 {lst_data['LST_Celsius'].mean():.2f}°C")
        except Exception as e:
            print(f"   ❌ {city_name}: 提取失败 - {e}")
    
    return results

# 批量提取
city_results = batch_extract_cities(cities, '2023-01-01', '2023-12-31')

### 步骤3: 对比分析

In [ ]:
# 创建对比表
comparison_data = []
for city_name, data in city_results.items():
    comparison_data.append({
        '城市': city_name,
        '平均温度(°C)': data['LST_Celsius'].mean(),
        '最高温度(°C)': data['LST_Celsius'].max(),
        '最低温度(°C)': data['LST_Celsius'].min(),
        '数据点数': len(data)
    })

comparison_df = pd.DataFrame(comparison_data)

print("📊 三城市地表温度对比 (2023年):")
print(comparison_df.to_string(index=False))

# 找出最热和最冷的城市
hottest = comparison_df.loc[comparison_df['平均温度(°C)'].idxmax()]
coldest = comparison_df.loc[comparison_df['平均温度(°C)'].idxmin()]

print(f"\n🔥 最热城市: {hottest['城市']} ({hottest['平均温度(°C)']:.2f}°C)")
print(f"❄️  最冷城市: {coldest['城市']} ({coldest['平均温度(°C)']:.2f}°C)")

---

## 📖 教程3: 数据可视化入门

**目标**: 快速可视化地表温度数据

**时间**: 1分钟

### 可视化1: 简单散点图

In [ ]:
import matplotlib.pyplot as plt

# 使用教程1的数据
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    lst_data['longitude'],
    lst_data['latitude'],
    c=lst_data['LST_Celsius'],
    cmap='RdYlBu_r',
    s=100,
    alpha=0.7,
    edgecolors='black',
    linewidths=0.5
)

plt.colorbar(scatter, label='温度 (°C)')
plt.xlabel('经度', fontsize=12)
plt.ylabel('纬度', fontsize=12)
plt.title('北京地表温度分布 (2023年平均)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ 空间分布图已生成！")

### 可视化2: 温度直方图

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(lst_data['LST_Celsius'], bins=20, edgecolor='black', alpha=0.7)
plt.axvline(lst_data['LST_Celsius'].mean(), color='red', 
           linestyle='--', linewidth=2, label=f"平均值: {lst_data['LST_Celsius'].mean():.2f}°C")
plt.xlabel('温度 (°C)', fontsize=12)
plt.ylabel('频数', fontsize=12)
plt.title('地表温度分布直方图', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("✅ 温度分布图已生成！")

### 可视化3: 城市对比柱状图

In [ ]:
# 使用教程2的数据
plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
plt.bar(comparison_df['城市'], comparison_df['平均温度(°C)'], color=colors, alpha=0.7)
plt.ylabel('平均温度 (°C)', fontsize=12)
plt.title('三城市地表温度对比 (2023年)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("✅ 城市对比图已生成！")

---

## 🎉 恭喜！

你已经完成了5分钟快速教程！

### 你学会了：

✅ 提取单个城市的LST数据
✅ 批量处理多个城市
✅ 数据可视化（空间分布、直方图、对比图）
✅ 保存数据（CSV、GeoJSON）

### 下一步：

1. 🚀 **提取你自己的数据**: 修改城市边界和时间范围
2. 📊 **尝试其他数据类型**: NDVI、降水、人口密度
3. 📚 **查看完整文档**: `COMPLETE_GUIDE.md`
4. 🔧 **探索高级功能**: 打开 `LST_Tools_Master.ipynb`

### 常用提示：

- 调整`spacing`参数来改变采样点密度
- 修改时间范围来分析不同时期
- 使用`quality_check()`检查数据质量
- 查看`COMPLETE_GUIDE.md`了解更多功能

---

**祝你研究顺利！** 🌍✨